# NHANES Mortality × Liver Fibrosis — Data Download & Preparation

This notebook downloads NHANES public-use data and linked mortality files for **all 10
continuous NHANES cycles** (1999-2000 through 2017-2018), merges the components, and saves
per-cohort parquet files for downstream analysis.

**Data sources:**
- NHANES SAS transport files (demographics, labs, body measures, blood pressure, lipids, glucose, smoking, elastography)
- NCHS Public-Use Linked Mortality Files (2019 release, follow-up through Dec 31, 2019)

**Note on file naming:** The NHANES file naming convention changed over time.
Cycles 1999-2004 use legacy lab file names (LAB18, L40, LAB25, L25, LAB13AM, L13AM, etc.);
cycles 2005+ use standardized names (BIOPRO, CBC, TRIGLY, GLU).

In [1]:
import os, hashlib
import requests
import numpy as np
import pandas as pd
import pyreadstat

DATA_DIR   = os.path.abspath(os.path.join('..', 'data'))
RAW_NHANES = os.path.join(DATA_DIR, 'raw', 'nhanes')
RAW_LMF    = os.path.join(DATA_DIR, 'raw', 'lmf')
DERIVED    = os.path.join(DATA_DIR, 'derived')
for d in [RAW_NHANES, RAW_LMF, DERIVED]:
    os.makedirs(d, exist_ok=True)

## Configuration

Each cycle defines: year, letter suffix, file-name overrides for legacy lab files,
and whether elastography (VCTE) data exists.

In [2]:
NHANES_BASE = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public'
LMF_BASE = 'https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality'

def _lmf_url(start, end):
    return f'{LMF_BASE}/NHANES_{start}_{end}_MORT_2019_PUBLIC.dat'

# File-name overrides for early cycles where lab files use legacy names.
# Keys are our standardized names; values are the actual XPT file prefixes.
COHORTS = {
    '1999-2000': {
        'year': 1999, 'suffix': '', 'has_elast': False,
        'lmf': _lmf_url(1999, 2000),
        'file_map': {'BIOPRO': 'LAB18', 'CBC': 'LAB25',
                     'TRIGLY': 'LAB13AM', 'GLU': 'LAB10AM'},
    },
    '2001-2002': {
        'year': 2001, 'suffix': 'B', 'has_elast': False,
        'lmf': _lmf_url(2001, 2002),
        'file_map': {'BIOPRO': 'L40', 'CBC': 'L25',
                     'TRIGLY': 'L13AM', 'GLU': 'L10AM'},
    },
    '2003-2004': {
        'year': 2003, 'suffix': 'C', 'has_elast': False,
        'lmf': _lmf_url(2003, 2004),
        'file_map': {'BIOPRO': 'L40', 'CBC': 'L25',
                     'TRIGLY': 'L13AM', 'GLU': 'L10AM'},
    },
    '2005-2006': {
        'year': 2005, 'suffix': 'D', 'has_elast': False,
        'lmf': _lmf_url(2005, 2006), 'file_map': {},
    },
    '2007-2008': {
        'year': 2007, 'suffix': 'E', 'has_elast': False,
        'lmf': _lmf_url(2007, 2008), 'file_map': {},
    },
    '2009-2010': {
        'year': 2009, 'suffix': 'F', 'has_elast': False,
        'lmf': _lmf_url(2009, 2010), 'file_map': {},
    },
    '2011-2012': {
        'year': 2011, 'suffix': 'G', 'has_elast': False,
        'lmf': _lmf_url(2011, 2012), 'file_map': {},
    },
    '2013-2014': {
        'year': 2013, 'suffix': 'H', 'has_elast': False,
        'lmf': _lmf_url(2013, 2014), 'file_map': {},
    },
    '2015-2016': {
        'year': 2015, 'suffix': 'I', 'has_elast': False,
        'lmf': _lmf_url(2015, 2016), 'file_map': {},
    },
    '2017-2018': {
        'year': 2017, 'suffix': 'J', 'has_elast': True,
        'lmf': _lmf_url(2017, 2018), 'file_map': {},
    },
}

NHANES_FILES = ['DEMO', 'BIOPRO', 'CBC', 'BMX', 'BPX', 'TRIGLY', 'GLU', 'SMQ']

## Download helpers

In [3]:
def download(url, dest):
    """Download with caching."""
    if os.path.exists(dest):
        return dest
    print(f'  GET {url}')
    r = requests.get(url, timeout=180); r.raise_for_status()
    with open(dest, 'wb') as f: f.write(r.content)
    print(f'    -> {os.path.basename(dest)} ({len(r.content):,} bytes, '
          f'MD5 {hashlib.md5(r.content).hexdigest()})')
    return dest

def read_xpt(path):
    try:
        df, _ = pyreadstat.read_xport(path)
    except UnicodeDecodeError:
        df, _ = pyreadstat.read_xport(path, encoding='latin1')
    return df

def _safe(s, to_type=int):
    s = s.strip().replace('.', '')
    if not s: return None
    try: return to_type(s)
    except ValueError: return None

def parse_lmf(path):
    rows = []
    with open(path) as f:
        for line in f:
            p = line.rstrip('\r\n').ljust(48)
            rows.append({
                'SEQN': _safe(p[0:14]),
                'ELIGSTAT':    _safe(p[14:15]),
                'MORTSTAT':    _safe(p[15:16]),
                'UCOD_LEADING':_safe(p[16:19]),
                'PERMTH_INT':  _safe(p[42:45], float),
                'PERMTH_EXM':  _safe(p[45:48], float),
            })
    return pd.DataFrame(rows)

## Download NHANES + LMF for all 10 cycles

In [4]:
raw = {}

for cycle, cfg in COHORTS.items():
    print(f'\n=== {cycle} ===')
    yr, sfx = cfg['year'], cfg['suffix']
    file_map = cfg.get('file_map', {})
    cdir = os.path.join(RAW_NHANES, cycle.replace('-','_'))
    os.makedirs(cdir, exist_ok=True)
    d = {}
    
    files_to_get = NHANES_FILES.copy()
    if cfg['has_elast']:
        files_to_get.append('LUX')
    
    for name in files_to_get:
        # Use file_map override if this cycle has a legacy name for this file
        actual_prefix = file_map.get(name, name)
        if sfx:
            fname = f'{actual_prefix}_{sfx}.xpt'
        else:
            fname = f'{actual_prefix}.xpt'
        url = f'{NHANES_BASE}/{yr}/DataFiles/{fname}'
        try:
            path = download(url, os.path.join(cdir, fname))
            d[name] = read_xpt(path)
            print(f'  {name} ({fname}): {len(d[name]):,} rows')
        except Exception as e:
            print(f'  {name} ({fname}): FAILED ({e})')
    
    # LMF
    lmf_path = download(cfg['lmf'],
                        os.path.join(RAW_LMF, os.path.basename(cfg['lmf'])))
    d['LMF'] = parse_lmf(lmf_path)
    print(f'  LMF: {len(d["LMF"]):,} rows')
    
    raw[cycle] = d

print(f'\nDownloaded {len(raw)} cycles')


=== 1999-2000 ===


  DEMO (DEMO.xpt): 9,965 rows
  BIOPRO (LAB18.xpt): 6,758 rows
  CBC (LAB25.xpt): 8,832 rows
  BMX (BMX.xpt): 9,282 rows
  BPX (BPX.xpt): 9,282 rows
  TRIGLY (LAB13AM.xpt): 3,915 rows
  GLU (LAB10AM.xpt): 3,267 rows


  SMQ (SMQ.xpt): 4,880 rows
  LMF: 9,965 rows

=== 2001-2002 ===
  DEMO (DEMO_B.xpt): 11,039 rows
  BIOPRO (L40_B.xpt): 7,445 rows
  CBC (L25_B.xpt): 9,929 rows
  BMX (BMX_B.xpt): 10,477 rows


  BPX (BPX_B.xpt): 10,477 rows
  TRIGLY (L13AM_B.xpt): 4,464 rows
  GLU (L10AM_B.xpt): 3,666 rows


  SMQ (SMQ_B.xpt): 5,411 rows
  LMF: 11,039 rows

=== 2003-2004 ===


...

  DEMO (DEMO_H.xpt): 10,175 rows
  BIOPRO (BIOPRO_H.xpt): 6,979 rows
  CBC (CBC_H.xpt): 9,422 rows
  BMX (BMX_H.xpt): 9,813 rows


  BPX (BPX_H.xpt): 9,813 rows
  TRIGLY (TRIGLY_H.xpt): 3,329 rows
  GLU (GLU_H.xpt): 3,329 rows
  SMQ (SMQ_H.xpt): 7,168 rows
  LMF: 10,175 rows

=== 2015-2016 ===
  DEMO (DEMO_I.xpt): 9,971 rows
  BIOPRO (BIOPRO_I.xpt): 6,744 rows


  CBC (CBC_I.xpt): 9,165 rows
  BMX (BMX_I.xpt): 9,544 rows
  BPX (BPX_I.xpt): 9,544 rows
  TRIGLY (TRIGLY_I.xpt): 3,191 rows
  GLU (GLU_I.xpt): 3,191 rows
  SMQ (SMQ_I.xpt): 7,001 rows
  LMF: 9,971 rows

=== 2017-2018 ===


  DEMO (DEMO_J.xpt): 9,254 rows
  BIOPRO (BIOPRO_J.xpt): 6,401 rows
  CBC (CBC_J.xpt): 8,366 rows


  BMX (BMX_J.xpt): 8,704 rows
  BPX (BPX_J.xpt): 8,704 rows
  TRIGLY (TRIGLY_J.xpt): 3,036 rows
  GLU (GLU_J.xpt): 3,036 rows
  SMQ (SMQ_J.xpt): 6,724 rows
  LUX (LUX_J.xpt): 6,401 rows
  LMF: 9,254 rows

Downloaded 10 cycles


## Merge components into analytic datasets

In [5]:
def _get_col(df, *candidates):
    """Return the first column name from candidates that exists in df."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

def merge_cohort(cycle, raw_d, has_elast):
    """Merge all components for one cohort."""
    if 'DEMO' not in raw_d:
        print(f'{cycle}: DEMO missing, skipping')
        return None
    df = raw_d['DEMO'].copy()
    
    # Labs: AST, ALT from BIOPRO; Platelets from CBC
    if 'BIOPRO' in raw_d:
        bdf = raw_d['BIOPRO']
        ast_col = _get_col(bdf, 'LBXSASSI', 'LBXSAS')
        alt_col = _get_col(bdf, 'LBXSATSI', 'LBXSAT')
        if ast_col and alt_col:
            bio = bdf[['SEQN', ast_col, alt_col]].rename(
                columns={ast_col: 'AST', alt_col: 'ALT'})
            df = df.merge(bio, on='SEQN', how='left')
        else:
            df['AST'] = np.nan; df['ALT'] = np.nan
    else:
        df['AST'] = np.nan; df['ALT'] = np.nan
    
    if 'CBC' in raw_d:
        cdf = raw_d['CBC']
        plt_col = _get_col(cdf, 'LBXPLTSI', 'LBXPLT')
        if plt_col:
            cbc = cdf[['SEQN', plt_col]].rename(columns={plt_col: 'PLATELETS'})
            df = df.merge(cbc, on='SEQN', how='left')
        else:
            df['PLATELETS'] = np.nan
    else:
        df['PLATELETS'] = np.nan
    
    # BMI
    if 'BMX' in raw_d:
        bmx = raw_d['BMX'][['SEQN','BMXBMI']]
        df = df.merge(bmx, on='SEQN', how='left')
    else:
        df['BMXBMI'] = np.nan
    
    # Blood pressure: average of available systolic readings
    if 'BPX' in raw_d:
        bpx = raw_d['BPX'][['SEQN'] + [c for c in raw_d['BPX'].columns
                                        if c.startswith('BPXSY')]].copy()
        sy_cols = [c for c in bpx.columns if c.startswith('BPXSY')]
        if sy_cols:
            bpx['SBP_MEAN'] = bpx[sy_cols].mean(axis=1)
            df = df.merge(bpx[['SEQN','SBP_MEAN']], on='SEQN', how='left')
        else:
            df['SBP_MEAN'] = np.nan
    else:
        df['SBP_MEAN'] = np.nan
    
    # LDL-C (fasting subsample)
    if 'TRIGLY' in raw_d and 'LBDLDL' in raw_d['TRIGLY'].columns:
        df = df.merge(raw_d['TRIGLY'][['SEQN','LBDLDL']], on='SEQN', how='left')
    else:
        df['LBDLDL'] = np.nan
    
    # Fasting plasma glucose
    if 'GLU' in raw_d and 'LBXGLU' in raw_d['GLU'].columns:
        df = df.merge(raw_d['GLU'][['SEQN','LBXGLU']], on='SEQN', how='left')
    else:
        df['LBXGLU'] = np.nan
    
    # Smoking: SMQ020=1 -> ever smoked 100+ cigarettes
    if 'SMQ' in raw_d and 'SMQ020' in raw_d['SMQ'].columns:
        smq = raw_d['SMQ'][['SEQN','SMQ020']].copy()
        smq['SMOKE_EVER'] = (smq['SMQ020'] == 1).astype(float)
        smq.loc[~smq['SMQ020'].isin([1,2]), 'SMOKE_EVER'] = np.nan
        df = df.merge(smq[['SEQN','SMOKE_EVER']], on='SEQN', how='left')
    else:
        df['SMOKE_EVER'] = np.nan
    
    # Elastography (2017-2018 only)
    if has_elast and 'LUX' in raw_d:
        lux = raw_d['LUX'][['SEQN','LUXSMED','LUXCAPM']].rename(
            columns={'LUXSMED':'LSM_KPA','LUXCAPM':'CAP_DBM'})
        df = df.merge(lux, on='SEQN', how='left')
    
    # Linked Mortality
    df = df.merge(raw_d['LMF'], on='SEQN', how='left')
    
    # Filter: eligible adults >= 18
    n0 = len(df)
    df = df[df['ELIGSTAT'] == 1].copy()
    df['AGE'] = df['RIDAGEYR']
    df = df[df['AGE'] >= 18].copy()
    df['FEMALE'] = (df['RIAGENDR'] == 2).astype(float)
    df['CYCLE'] = cycle
    
    # Report data availability
    n = len(df)
    ast_n = df['AST'].notna().sum()
    ldl_n = df['LBDLDL'].notna().sum()
    glu_n = df['LBXGLU'].notna().sum()
    print(f'{cycle}: {n0} -> {n} eligible adults '
          f'(AST: {ast_n}, LDL: {ldl_n}, GLU: {glu_n})')
    return df

In [6]:
for cycle, cfg in COHORTS.items():
    df = merge_cohort(cycle, raw[cycle], cfg['has_elast'])
    if df is None:
        continue
    out_path = os.path.join(DERIVED, f'{cycle.replace("-","_")}.parquet')
    df.to_parquet(out_path, index=False)
    print(f'  -> {out_path} ({os.path.getsize(out_path)/1e6:.1f} MB)\n')

print('Done. Parquet files ready in data/derived/')

1999-2000: 9965 -> 5445 eligible adults (AST: 4619, LDL: 1984, GLU: 2277)


  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/1999_2000.parquet (3.4 MB)

2001-2002: 11039 -> 5987 eligible adults (AST: 5209, LDL: 2327, GLU: 2642)
  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/2001_2002.parquet (0.4 MB)

2003-2004: 10122 -> 5610 eligible adults (AST: 4956, LDL: 2341, GLU: 2417)
  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/2003_2004.parquet (0.3 MB)



2005-2006: 10348 -> 5561 eligible adults (AST: 4899, LDL: 2336, GLU: 2424)
  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/2005_2006.parquet (0.3 MB)

2007-2008: 10149 -> 6219 eligible adults (AST: 5559, LDL: 2678, GLU: 2751)
  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/2007_2008.parquet (0.3 MB)

2009-2010: 10537 -> 6510 eligible adults (AST: 5947, LDL: 2854, GLU: 2929)


  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/2009_2010.parquet (0.3 MB)

2011-2012: 9756 -> 5849 eligible adults (AST: 5142, LDL: 2515, GLU: 2595)
  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/2011_2012.parquet (0.3 MB)

2013-2014: 10175 -> 6100 eligible adults (AST: 5616, LDL: 2662, GLU: 2723)


  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/2013_2014.parquet (0.3 MB)

2015-2016: 9971 -> 5974 eligible adults (AST: 5374, LDL: 2335, GLU: 2568)
  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/2015_2016.parquet (0.3 MB)

2017-2018: 9254 -> 5809 eligible adults (AST: 5107, LDL: 2451, GLU: 2520)


  -> /home/abie/ai_assisted_us_health_data_analysis/data/derived/2017_2018.parquet (0.3 MB)

Done. Parquet files ready in data/derived/


In [7]:
# Summary of all cycles
summary = []
for f in sorted(os.listdir(DERIVED)):
    if not f.endswith('.parquet'):
        continue
    df = pd.read_parquet(os.path.join(DERIVED, f))
    cycle = f.replace('.parquet','').replace('_','-')
    summary.append({
        'Cycle': cycle,
        'N': len(df),
        'Deaths': int((df['MORTSTAT']==1).sum()),
        'FU range': f"{df['PERMTH_EXM'].min():.0f}\u2013{df['PERMTH_EXM'].max():.0f}",
        'AST avail': int(df['AST'].notna().sum()),
        'LDL avail': int(df['LBDLDL'].notna().sum()),
        'GLU avail': int(df['LBXGLU'].notna().sum()),
        'Has LSM': 'LSM_KPA' in df.columns,
    })

pd.DataFrame(summary)

,Cycle,N,Deaths,FU range,AST avail,LDL avail,GLU avail,Has LSM
0,1999-2000,5445,1675,1–249,4619,1984,2277,False
1,2001-2002,5987,1624,0–229,5209,2327,2642,False
2,2003-2004,5610,1420,0–205,4956,2341,2417,False
3,2005-2006,5561,1027,1–180,4899,2336,2424,False
4,2007-2008,6219,1126,0–159,5559,2678,2751,False
5,2009-2010,6510,861,1–135,5947,2854,2929,False
6,2011-2012,5849,628,0–113,5142,2515,2595,False
7,2013-2014,6100,467,0–85,5616,2662,2723,False
8,2015-2016,5974,276,0–61,5374,2335,2568,False
9,2017-2018,5809,145,0–37,5107,2451,2520,True
